# Object Detection: Localization & Classification

Reach for this when you need: 
- Reference for localization tasks (where is the object?).
- To understand Bounding Box formats (`xyxy` vs `xywh`).
- Implementing post-processing like NMS (Non-Maximum Suppression).

In [1]:
import torch
import torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.ops import nms, box_iou

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. Faster R-CNN Template

| Feature | Description | Usage |
| :--- | :--- | :--- |
| `Backbone` | Extract features (e.g., ResNet) | Foundation for localization |
| `RPN` | Propose regions | Generates candidate boxes |
| `ROI Pool` | Crop features for boxes | Classifying proposed regions |

In [ ]:
# Load pretrained detector
model = fasterrcnn_resnet50_fpn(weights='DEFAULT').to(device).eval()

img = torch.randn(1, 3, 400, 400).to(device)
with torch.no_grad():
    predictions = model(img)

# predictions[0] keys: 'boxes', 'labels', 'scores'
boxes = predictions[0]['boxes']
scores = predictions[0]['scores']

## 2. Bounding Box Post-processing

**NMS (Non-Maximum Suppression)**
Filters out overlapping boxes for the same object.

✅ **Use when**: Cleaning raw model outputs before visualization/evaluation.
❌ **Don't use when**: You have low-confidence thresholds and multiple overlapping OBJECTS (e.g. dense crowds).

In [ ]:
# Filter by confidence score
high_conf = scores > 0.8
filtered_boxes = boxes[high_conf]
filtered_scores = scores[high_conf]

# Apply NMS (iou_threshold=0.5 is standard)
keep_indices = nms(filtered_boxes, filtered_scores, iou_threshold=0.5)
final_boxes = filtered_boxes[keep_indices]

## 3. Intersection over Union (IoU)
Calculates overlap between two boxes. 

| Value | Meaning |
| :--- | :--- |
| 0 | No overlap |
| 1 | Perfect overlap |
| >0.5 | Reasonably good match |

In [2]:
box1 = torch.tensor([[10, 10, 50, 50]]) # [x1, y1, x2, y2]
box2 = torch.tensor([[15, 15, 55, 55]])
iou = box_iou(box1, box2)
print(f"IoU Score: {iou.item()}")

IoU Score: 0.6202531456947327


### Common Pitfalls
- **Box Formats**: PyTorch standard is `[x1, y1, x2, y2]`. Many datasets like COCO use `[x, y, w, h]`. ALWAYS convert before passing to `torchvision` ops.
- **Normalized Coordinates**: Ensure your boxes match the image scale (pixels vs. [0, 1]). Detection models usually expect pixel coordinates.
- **Training Mode**: Detection models often require a different dict structure `{boxes: ..., labels: ...}` in `model.train()`.

### Key Takeaways
- `Faster R-CNN` is 2-stage (slow but accurate); `YOLO/SSD` is 1-stage (fast for real-time).
- `NMS` is the critical final step for almost every detection pipeline.
- IoU is the standard metric for evaluation (`mAP` depends on IoU thresholds).